In [1]:
from datasets import load_dataset
from transformers import AutoImageProcessor,AutoFeatureExtractor, AutoModelForImageClassification, Trainer, TrainingArguments
import torch
from torchvision import transforms as T
import matplotlib.pyplot as plt

d:\PycharmProjects\deeplearning_25spr\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 加载数据集

In [22]:
# 加载数据集
print("Loading dataset...")
dataset_path = "../../data/touhou_split"
dataset = load_dataset("imagefolder", data_dir=dataset_path)
# print(dataset)
# print(dataset["train"][0].keys())

Loading dataset...


In [3]:
# 转换label
labels = dataset["train"].features["label"].names
label2id, id2label = dict(), dict()
for i, label in enumerate(labels):
    label2id[label] = i
    id2label[i] = label

## 数据预处理

In [4]:
# 加载特征提取器和模型
image_processor = AutoImageProcessor.from_pretrained(
    "google/vit-base-patch16-224", use_fast=True
)
feature_extractor = AutoFeatureExtractor.from_pretrained("google/vit-base-patch16-224")
train_transform = T.Compose(
    [
        T.ToTensor(),
        T.RandomResizedCrop(
            size=(256, 256),
            scale=(0.75, 1.0),
            ratio=(0.75, 1.33),
            interpolation=T.InterpolationMode.BICUBIC,
        ),
        T.RandomHorizontalFlip(),
        T.ColorJitter(brightness=(0.9, 1.1)),
        T.RandomRotation(degrees=45),
        T.RandomErasing(p=0.05, value="random"),
        # T.Normalize(mean=data_config["mean"], std=data_config["std"])
    ]
)
normalize = T.Normalize(mean=image_processor.image_mean, std=image_processor.image_std)
size = (
    image_processor.size["shortest_edge"]
    if "shortest_edge" in image_processor.size
    else (image_processor.size["height"], image_processor.size["width"])
)
_transforms = T.Compose([T.RandomResizedCrop(size), T.ToTensor(), normalize])

d:\PycharmProjects\deeplearning_25spr\.venv\Lib\site-packages\transformers\models\vit\feature_extraction_vit.py:28: FutureWarning: The class ViTFeatureExtractor is deprecated and will be removed in version 5 of Transformers. Please use ViTImageProcessor instead.
  warnings.warn(


In [5]:
# 定义图片预处理函数

def transforms(examples):
    examples["pixel_values"] = [_transforms(img.convert("RGB")) for img in examples["image"]]
    del examples["image"]
    return examples


def transform(examples):
    inputs = feature_extractor([x for x in examples["image"]], return_tensors="pt")
    inputs["labels"] = examples["label"]
    return inputs

In [35]:
# precessed_dataset = dataset.map(transforms, batched=True, batch_size=32)
processed_dataset = dataset.with_transform(transforms)
from transformers import DefaultDataCollator

data_collator = DefaultDataCollator(return_tensors="pt")

## 评估

In [7]:
# %pip install evaluate

In [29]:
import evaluate
import numpy as np

accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=predictions, references=labels)

## 加载模型

In [30]:
model = AutoModelForImageClassification.from_pretrained(
    "google/vit-base-patch16-224-in21k", num_labels=len(labels)
    , id2label=id2label, label2id=label2id
)

Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224-in21k and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [10]:
# %pip install accelerate==0.26.0

In [32]:
processed_dataset['train'][0]

{'pixel_values': tensor([[[-0.7255, -0.7412, -0.7490,  ..., -0.8824, -0.8902, -0.8980],
          [-0.6941, -0.7020, -0.7176,  ..., -0.8824, -0.8902, -0.8902],
          [-0.6157, -0.6235, -0.6392,  ..., -0.8745, -0.8824, -0.8902],
          ...,
          [ 0.2706,  0.2784,  0.3804,  ...,  0.8196,  0.8431,  0.7804],
          [ 0.2706,  0.2706,  0.2706,  ...,  0.8275,  0.8745,  0.8667],
          [ 0.2706,  0.2706,  0.2627,  ...,  0.8353,  0.8824,  0.8353]],
 
         [[-0.4902, -0.4980, -0.5059,  ..., -0.5765, -0.5686, -0.5765],
          [-0.4510, -0.4588, -0.4745,  ..., -0.5608, -0.5686, -0.5686],
          [-0.3882, -0.3961, -0.4118,  ..., -0.5451, -0.5529, -0.5529],
          ...,
          [-0.3804, -0.3882, -0.4039,  ...,  0.6078,  0.6392,  0.5843],
          [-0.3804, -0.3804, -0.3804,  ...,  0.6235,  0.6706,  0.6627],
          [-0.3882, -0.3882, -0.3882,  ...,  0.6392,  0.6863,  0.6314]],
 
         [[ 0.2235,  0.2078,  0.2078,  ...,  0.2314,  0.2392,  0.2471],
          [ 

In [36]:
training_args = TrainingArguments(
    output_dir="touhou_cls_vit",
    remove_unused_columns=False,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=50,
    warmup_ratio=0.1,
    logging_steps=10,
    metric_for_best_model="accuracy",
    report_to="tensorboard",
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=processed_dataset["train"],
    eval_dataset=processed_dataset["validation"],
    processing_class=image_processor,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)


d:\PycharmProjects\deeplearning_25spr\.venv\Lib\site-packages\transformers\training_args.py:1594: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [37]:
train_results = trainer.train()
trainer.save_model()
trainer.log_metrics("train", train_results.metrics)
trainer.save_metrics("train", train_results.metrics)
trainer.save_state()

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 